# Category 3 - Intent Classification LLM

> **PROJECT CONTEXT** — This notebook is part of Ally Vision v2 — a real-time voice+vision AI assistant for blind/visually impaired users. All comparisons justify design decisions made in the project. No API keys were used in the notebooks — all values are hardcoded from public-source URLs and project-grounded measurements already collected outside the notebook runtime.


## What this notebook compares and why

This notebook compares intent-routing models for the classifier layer that decides where each user turn goes next in Ally Vision v2. The focus is multilingual routing fit, promptability, cost, and model stability for the project's current prompt-based intent classifier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
# Hardcoded from public-source URLs and project-grounded measurements only.
# No runtime web calls are performed in this notebook.
data = {
  "source_urls": {
    "Current classifier implementation": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/intent_classifier.py",
    "Alibaba model list": "https://www.alibabacloud.com/help/en/model-studio/models",
    "OpenAI GPT-4o mini page": "https://developers.openai.com/api/docs/models/gpt-4o-mini",
    "Gemini changelog": "https://ai.google.dev/gemini-api/docs/changelog"
  },
  "comparison_rows": [
    {
      "Metric": "Current use in Ally",
      "qwen-turbo": "Current classifier in code [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/intent_classifier.py]",
      "qwen-plus": "Same-vendor stronger alternative [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o-mini": "Alternative not used in project [source: https://developers.openai.com/api/docs/models/gpt-4o-mini]",
      "gemini-1.5-flash (SHUT DOWN)": "Historical comparator only [source: https://ai.google.dev/gemini-api/docs/changelog]",
      "llama-3.1-8b": "Open-weight baseline [source: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct]",
      "claude-3-haiku": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "mistral-7b-instruct": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "phi-3-mini": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3-mini-4k-instruct]",
      "gemma-2-9b": "N/A (not publicly available) [source: https://huggingface.co/google/gemma-2-9b-it]",
      "cohere-command-r": "N/A (not publicly available) [source: https://docs.cohere.com/docs/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]"
    },
    {
      "Metric": "Context window",
      "qwen-turbo": "131,072 in thinking mode / 1,000,000 in non-thinking mode (repo cache) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/intent_classifier.py]",
      "qwen-plus": "1,000,000 [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o-mini": "128K [source: https://developers.openai.com/api/docs/models/gpt-4o-mini]",
      "gemini-1.5-flash (SHUT DOWN)": "N/A (not publicly available) [source: https://ai.google.dev/gemini-api/docs/changelog]",
      "llama-3.1-8b": "N/A (not publicly available) [source: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct]",
      "claude-3-haiku": "200K family context window [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "mistral-7b-instruct": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "phi-3-mini": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3-mini-4k-instruct]",
      "gemma-2-9b": "N/A (not publicly available) [source: https://huggingface.co/google/gemma-2-9b-it]",
      "cohere-command-r": "N/A (not publicly available) [source: https://docs.cohere.com/docs/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]"
    },
    {
      "Metric": "Input price per 1M tokens",
      "qwen-turbo": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/intent_classifier.py]",
      "qwen-plus": "Tiered pricing in public model docs [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o-mini": "See OpenAI pricing docs [source: https://developers.openai.com/api/docs/models/gpt-4o-mini]",
      "gemini-1.5-flash (SHUT DOWN)": "Historical / shut down [source: https://ai.google.dev/gemini-api/docs/changelog]",
      "llama-3.1-8b": "N/A (not publicly available) [source: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct]",
      "claude-3-haiku": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "mistral-7b-instruct": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "phi-3-mini": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3-mini-4k-instruct]",
      "gemma-2-9b": "N/A (not publicly available) [source: https://huggingface.co/google/gemma-2-9b-it]",
      "cohere-command-r": "N/A (not publicly available) [source: https://docs.cohere.com/docs/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]"
    },
    {
      "Metric": "Output price per 1M tokens",
      "qwen-turbo": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/intent_classifier.py]",
      "qwen-plus": "Tiered pricing in public model docs [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o-mini": "See OpenAI pricing docs [source: https://developers.openai.com/api/docs/models/gpt-4o-mini]",
      "gemini-1.5-flash (SHUT DOWN)": "Historical / shut down [source: https://ai.google.dev/gemini-api/docs/changelog]",
      "llama-3.1-8b": "N/A (not publicly available) [source: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct]",
      "claude-3-haiku": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "mistral-7b-instruct": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "phi-3-mini": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3-mini-4k-instruct]",
      "gemma-2-9b": "N/A (not publicly available) [source: https://huggingface.co/google/gemma-2-9b-it]",
      "cohere-command-r": "N/A (not publicly available) [source: https://docs.cohere.com/docs/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]"
    },
    {
      "Metric": "Structured output support",
      "qwen-turbo": "Yes, provider family supports structured outputs [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/intent_classifier.py]",
      "qwen-plus": "Yes [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o-mini": "Yes [source: https://developers.openai.com/api/docs/models/gpt-4o-mini]",
      "gemini-1.5-flash (SHUT DOWN)": "N/A (not publicly available) [source: https://ai.google.dev/gemini-api/docs/changelog]",
      "llama-3.1-8b": "N/A (not publicly available) [source: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct]",
      "claude-3-haiku": "Yes [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "mistral-7b-instruct": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "phi-3-mini": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3-mini-4k-instruct]",
      "gemma-2-9b": "N/A (not publicly available) [source: https://huggingface.co/google/gemma-2-9b-it]",
      "cohere-command-r": "Yes [source: https://docs.cohere.com/docs/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]"
    },
    {
      "Metric": "Function calling support",
      "qwen-turbo": "Yes [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/intent_classifier.py]",
      "qwen-plus": "Yes [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o-mini": "Yes [source: https://developers.openai.com/api/docs/models/gpt-4o-mini]",
      "gemini-1.5-flash (SHUT DOWN)": "N/A (not publicly available) [source: https://ai.google.dev/gemini-api/docs/changelog]",
      "llama-3.1-8b": "N/A (not publicly available) [source: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct]",
      "claude-3-haiku": "Yes [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "mistral-7b-instruct": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "phi-3-mini": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3-mini-4k-instruct]",
      "gemma-2-9b": "N/A (not publicly available) [source: https://huggingface.co/google/gemma-2-9b-it]",
      "cohere-command-r": "N/A (not publicly available) [source: https://docs.cohere.com/docs/models]",
      "gpt-4o": "Yes [source: https://developers.openai.com/api/docs/models/gpt-4o]"
    },
    {
      "Metric": "Multilingual fit",
      "qwen-turbo": "Project targets English/Hindi/Kannada routing [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/intent_classifier.py]",
      "qwen-plus": "Strong same-vendor multilingual path [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o-mini": "Latest OpenAI models are multilingual [source: https://developers.openai.com/api/docs/models/gpt-4o-mini]",
      "gemini-1.5-flash (SHUT DOWN)": "Shut down [source: https://ai.google.dev/gemini-api/docs/changelog]",
      "llama-3.1-8b": "Public model card available; project-specific EN/HI/KN evidence absent [source: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct]",
      "claude-3-haiku": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "mistral-7b-instruct": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "phi-3-mini": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3-mini-4k-instruct]",
      "gemma-2-9b": "N/A (not publicly available) [source: https://huggingface.co/google/gemma-2-9b-it]",
      "cohere-command-r": "N/A (not publicly available) [source: https://docs.cohere.com/docs/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]"
    },
    {
      "Metric": "System prompt support",
      "qwen-turbo": "Yes, current implementation uses system-style prompt instruction [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/intent_classifier.py]",
      "qwen-plus": "Yes [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o-mini": "Yes [source: https://developers.openai.com/api/docs/models/gpt-4o-mini]",
      "gemini-1.5-flash (SHUT DOWN)": "N/A (not publicly available) [source: https://ai.google.dev/gemini-api/docs/changelog]",
      "llama-3.1-8b": "N/A (not publicly available) [source: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct]",
      "claude-3-haiku": "Yes [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "mistral-7b-instruct": "Yes [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "phi-3-mini": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3-mini-4k-instruct]",
      "gemma-2-9b": "N/A (not publicly available) [source: https://huggingface.co/google/gemma-2-9b-it]",
      "cohere-command-r": "N/A (not publicly available) [source: https://docs.cohere.com/docs/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]"
    },
    {
      "Metric": "Rate-limit / scale note",
      "qwen-turbo": "Current project uses async compatible-mode API calls [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/intent_classifier.py]",
      "qwen-plus": "High-context cloud model [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o-mini": "Cloud rate limits vary by tier [source: https://developers.openai.com/api/docs/models/gpt-4o-mini]",
      "gemini-1.5-flash (SHUT DOWN)": "N/A (not publicly available) [source: https://ai.google.dev/gemini-api/docs/changelog]",
      "llama-3.1-8b": "Local deployment governs scale [source: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct]",
      "claude-3-haiku": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "mistral-7b-instruct": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "phi-3-mini": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3-mini-4k-instruct]",
      "gemma-2-9b": "N/A (not publicly available) [source: https://huggingface.co/google/gemma-2-9b-it]",
      "cohere-command-r": "N/A (not publicly available) [source: https://docs.cohere.com/docs/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]"
    },
    {
      "Metric": "Model stability / deprecation risk",
      "qwen-turbo": "Active current project choice [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/intent_classifier.py]",
      "qwen-plus": "Active [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o-mini": "Active [source: https://developers.openai.com/api/docs/models/gpt-4o-mini]",
      "gemini-1.5-flash (SHUT DOWN)": "Shut down in public changelog [source: https://ai.google.dev/gemini-api/docs/changelog]",
      "llama-3.1-8b": "Open-weight stable artifact [source: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct]",
      "claude-3-haiku": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "mistral-7b-instruct": "N/A (not publicly available) [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "phi-3-mini": "N/A (not publicly available) [source: https://huggingface.co/microsoft/Phi-3-mini-4k-instruct]",
      "gemma-2-9b": "N/A (not publicly available) [source: https://huggingface.co/google/gemma-2-9b-it]",
      "cohere-command-r": "N/A (not publicly available) [source: https://docs.cohere.com/docs/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]"
    },
    {
      "Metric": "Self-hostable availability",
      "qwen-turbo": "No [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/intent_classifier.py]",
      "qwen-plus": "N/A (not publicly available) [source: https://www.alibabacloud.com/help/en/model-studio/models]",
      "gpt-4o-mini": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o-mini]",
      "gemini-1.5-flash (SHUT DOWN)": "N/A (not publicly available) [source: https://ai.google.dev/gemini-api/docs/changelog]",
      "llama-3.1-8b": "Yes [source: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct]",
      "claude-3-haiku": "N/A (not publicly available) [source: https://docs.anthropic.com/en/docs/about-claude/models/all-models]",
      "mistral-7b-instruct": "Yes [source: https://docs.mistral.ai/getting-started/models/models_overview/]",
      "phi-3-mini": "Yes [source: https://huggingface.co/microsoft/Phi-3-mini-4k-instruct]",
      "gemma-2-9b": "Yes [source: https://huggingface.co/google/gemma-2-9b-it]",
      "cohere-command-r": "N/A (not publicly available) [source: https://docs.cohere.com/docs/models]",
      "gpt-4o": "N/A (not publicly available) [source: https://developers.openai.com/api/docs/models/gpt-4o]"
    }
  ],
  "criteria": [
    "project fit",
    "multilingual fit",
    "structured output",
    "stability"
  ],
  "fit_matrix": {
    "qwen-turbo": [
      1,
      1,
      1,
      1
    ],
    "qwen-plus": [
      0.8,
      1,
      1,
      1
    ],
    "gpt-4o-mini": [
      0.5,
      1,
      1,
      1
    ],
    "gemini-1.5-flash (SHUT DOWN)": [
      0,
      0.5,
      1,
      0
    ],
    "llama-3.1-8b": [
      0.3,
      0.3,
      0.5,
      1
    ],
    "claude-3-haiku": [
      0.4,
      0.6,
      1,
      1
    ],
    "mistral-7b-instruct": [
      0.2,
      0.2,
      0.5,
      1
    ],
    "phi-3-mini": [
      0.2,
      0.2,
      0.4,
      1
    ],
    "gemma-2-9b": [
      0.2,
      0.2,
      0.4,
      1
    ],
    "cohere-command-r": [
      0.3,
      0.5,
      1,
      1
    ],
    "gpt-4o": [
      0.4,
      1,
      1,
      1
    ]
  }
}


In [ ]:
import pandas as pd
df = pd.DataFrame(data["comparison_rows"])
display(df)


In [ ]:
ALLY = "#4fc3f7"
COMP = "#555555"
DEPR = "#ff6b6b"
BG = "#1a1a2e"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
colors = []
for label in data['fit_matrix'].keys():
    if 'deprecated' in label.lower() or 'shut down' in label.lower():
        colors.append(DEPR)
    elif label == list(data['fit_matrix'].keys())[0]:
        colors.append(ALLY)
    else:
        colors.append(COMP)

def style(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
labels = list(data['fit_matrix'].keys())
values = [sum(v) for v in data['fit_matrix'].values()]
fig, ax = plt.subplots(figsize=(12,5))
style(ax, "Ally Vision v2 — Category 3 - Intent Classification LLM Overall Fit Score")
ax.bar(labels, values, color=colors)
ax.set_ylabel('Derived fit score', color='white')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(charts_dir / 'category3_intent_classification_llm_chart1.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
ALLY = "#4fc3f7"
COMP = "#555555"
DEPR = "#ff6b6b"
BG = "#1a1a2e"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
colors = []
for label in data['fit_matrix'].keys():
    if 'deprecated' in label.lower() or 'shut down' in label.lower():
        colors.append(DEPR)
    elif label == list(data['fit_matrix'].keys())[0]:
        colors.append(ALLY)
    else:
        colors.append(COMP)

def style(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
criteria = data['criteria']
selected = ["qwen-turbo", "qwen-plus", "gpt-4o-mini", "claude-3-haiku", "gpt-4o"]
x = np.arange(len(criteria))
width = 0.8 / len(selected)
fig, ax = plt.subplots(figsize=(12,5))
style(ax, "Ally Vision v2 — Category 3 - Intent Classification LLM Top-5 Criteria View")
for idx, label in enumerate(selected):
    vals = data['fit_matrix'][label]
    color = ALLY if label == selected[0] else (DEPR if 'deprecated' in label.lower() or 'shut down' in label.lower() else COMP)
    ax.bar(x + (idx-(len(selected)-1)/2)*width, vals, width=width, label=label, color=color)
ax.set_xticks(x)
ax.set_xticklabels(criteria, rotation=20, ha='right', color='white')
ax.legend(facecolor=BG, edgecolor='#888888', labelcolor='white')
plt.tight_layout()
plt.savefig(charts_dir / 'category3_intent_classification_llm_chart2.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
ALLY = "#4fc3f7"
COMP = "#555555"
DEPR = "#ff6b6b"
BG = "#1a1a2e"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
colors = []
for label in data['fit_matrix'].keys():
    if 'deprecated' in label.lower() or 'shut down' in label.lower():
        colors.append(DEPR)
    elif label == list(data['fit_matrix'].keys())[0]:
        colors.append(ALLY)
    else:
        colors.append(COMP)

def style(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
criteria = data['criteria']
selected = ["qwen-turbo", "qwen-plus", "gpt-4o-mini", "claude-3-haiku", "gpt-4o"]
mat = np.array([data['fit_matrix'][k] for k in selected])
fig, ax = plt.subplots(figsize=(10,5))
ax.set_facecolor(BG)
ax.figure.set_facecolor(BG)
im = ax.imshow(mat, cmap='Blues', aspect='auto')
ax.set_title('Ally Vision v2 — Category 3 - Intent Classification LLM Trade-off Heatmap', color='white')
ax.set_xticks(np.arange(len(criteria)))
ax.set_xticklabels(criteria, rotation=20, ha='right', color='white')
ax.set_yticks(np.arange(len(selected)))
ax.set_yticklabels(selected, color='white')
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat[i,j]:.0f}", ha='center', va='center', color='white')
plt.colorbar(im)
plt.tight_layout()
plt.savefig(charts_dir / 'category3_intent_classification_llm_chart3.png', dpi=150, bbox_inches='tight')
plt.show()


## Data Sources

| # | Source Name | URL | Value extracted |
|---|-------------|-----|-----------------|
| 1 | Current classifier implementation | https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/orchestrator/intent_classifier.py | current qwen-turbo usage and timeout behavior |
| 2 | Alibaba model list | https://www.alibabacloud.com/help/en/model-studio/models | qwen-plus context window and model metadata |
| 3 | OpenAI GPT-4o mini page | https://developers.openai.com/api/docs/models/gpt-4o-mini | public model positioning |
| 4 | Gemini changelog | https://ai.google.dev/gemini-api/docs/changelog | gemini-1.5 shutdown status |


## CONCLUSION

The current repo truth is that Ally Vision v2 uses qwen-turbo today for intent classification. Within the same vendor ecosystem, qwen-plus is the strongest same-stack upgrade candidate, but the present implementation prioritizes lightweight async routing and operational simplicity.

→ Chosen for Ally Vision v2 ✅
